# I. Installing Dependencies

In [16]:
import re
import json
import uuid
import sqlite3
from pathlib import Path

import base64
import aiosqlite
import httpx
import gradio as gr
from openai import AsyncOpenAI

In [17]:
from google.colab import userdata

# II. Setting-up Client

In [18]:
OpenAI_API_KEY = userdata.get("OPENAI_API")

In [19]:
client = AsyncOpenAI(api_key=OpenAI_API_KEY)

In [20]:
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks"

In [21]:
DB = f"{BASE_PATH}/chats.db"

In [22]:
IMG_DIR = Path(f"{BASE_PATH}/OpenAI_Images")
IMG_DIR.mkdir(exist_ok=True)

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# III. Table Setup

In [24]:
with sqlite3.connect(DB) as conn:
    conn.execute("""
    CREATE TABLE IF NOT EXISTS conversations(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        messages TEXT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

In [25]:
IMG_RE = re.compile(r"!\[[^\]]*\]\(([^)]+)\)")

# IV. Helper Functions

In [26]:
def save_image(b64:str)->Path:
  path = IMG_DIR / f"{uuid.uuid4().hex}.png"
  path.write_bytes(base64.b64decode(b64))
  return path

In [27]:
async def download_image(url:str)->Path:
    path = IMG_DIR/f"{uuid.uuid4().hex}.png"
    async with httpx.AsyncClient(timeout=60) as http:
        r = await http.get(url)
        r.raise_for_status()
    path.write_bytes(r.content)
    return path

In [28]:
def purge_image(history:list[dict])->None:
    for msg in history:
        content = msg.get("content")
        if not isinstance(content, str):
            continue
        for ref in IMG_RE.findall(content):
            p = Path(ref)
            try:
                if p.resolve().is_relative_to(IMG_DIR.resolve()):
                    p.unlink(missing_ok=True)
            except (ValueError, OSError):
                ... # Path is outside our directory, or already deleted!

# V. Chat Handler

In [37]:
async def chat(message, history):
   if not message.strip():
    return history, "" # record the user's text turn first
    history = history + [{"role": "user", "content": message}]
    if message.lower().startswith("/image "):
      prompt = message[7:].strip()
      result = await client.images.generate( model="gpt-image-1", prompt=prompt, size="1024x1024" )
      local_path = save_image(result.data[0].b64_json) # media format -> Gradio serves & displays the file (no markdown)
      assistant_msg = { "role": "assistant", "content": {"path": str(local_path), "alt_text": prompt}, }
    else: # only send string-content messages to the model; skip image dicts
      msgs = [m for m in history if isinstance(m["content"], str)]
      response = await client.chat.completions.create( model="gpt-5.4-mini", messages=msgs )
      assistant_msg = { "role": "assistant", "content": response.choices[0].message.content, }
      history = history + [assistant_msg]
    return history, ""

# VI. Persistence

In [30]:
async def list_conversations():
    async with aiosqlite.connect(DB) as db:
        cursor = await db.execute(
            "SELECT id, title FROM conversations ORDER BY created_at DESC"
        )
        rows = await cursor.fetchall()
    return gr.Dropdown(choices=[(r[1], r[0]) for r in rows], value=None)

In [31]:
async def save_conversation(history, title):
    if not history:
        return await list_conversations(), "Nothing to save."
    title = (title or "").strip() or f"Chat - {len(history)//2} turns"
    async with aiosqlite.connect(DB) as db:
        await db.execute(
          "INSERT INTO conversations (title, messages) VALUES (?, ?)",
          (title, json.dumps(history)),
        )
        await db.commit()
    return await list_conversations(), f"Saved: {title}"

In [32]:
async def load_conversation(conv_id):
    if conv_id is None:
        return [], "Select a conversation first."
    async with aiosqlite.connect(DB) as db:
        cursor = await db.execute(
            "SELECT title, messages FROM conversations WHERE id=?", (conv_id,)
        )
        row = await cursor.fetchone()
    if row is None:
        return [], "Not found"
    return json.loads(row[1]), f"Loaded: {row[0]}"

In [33]:
async def delete_conversation(conv_id):
    if conv_id is None:
        return await list_conversations(), "Select a conversation first."

    async with aiosqlite.connect(DB) as db:
        cursor = await db.execute(
            "SELECT messages FROM conversations WHERE id=?", (conv_id,)
        )
        row = await cursor.fetchone()
        if row:
            purge_image(json.loads(row[0]))

        await db.execute("DELETE FROM conversations WHERE id=?",  (conv_id,))
        await db.commit()
    return await list_conversations(), "Deleted!"


# VII. UI

In [34]:
with gr.Blocks(title="OpenAI Chatbot") as demo:
    gr.Markdown("# OpenAI Chatbott\nChat normally, or `/image <prompt>` for DALL-E.")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages", height=500)
            msg = gr.Textbox(placeholder="Message or /imge <prompt>...", show_label=False)

        with gr.Column(scale=1):
            gr.Markdown("### Saved Chats")
            saved = gr.Dropdown(label="Conversations", choices=[], interactive=True)
            title_in = gr.Textbox(label="Title (for saving)", placeholder="optional")
            with gr.Row():
                save_btn = gr.Button("💾 Save")
                load_btn = gr.Button("📁 Load")
            with gr.Row():
                del_btn = gr.Button("🗑️ Delete", variant="stop")
                clear_btn = gr.Button("✨ New")
            status = gr.Markdown("")

    msg.submit(chat, [msg, chatbot], [chatbot, msg])
    save_btn.click(save_conversation, [chatbot, title_in], [saved, status])
    load_btn.click(load_conversation, [saved], [chatbot, status])
    del_btn.click(delete_conversation, [saved], [saved, status])
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, status])
    demo.load(list_conversations, outputs=[saved])

/tmp/ipykernel_3455/610843271.py:6: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", height=500)


In [36]:
demo.launch(allowed_paths=[str(IMG_DIR.resolve())])

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f648a80a9353b73e1d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
